# Bar strategies for a trading platform: breakout, mean reversion and calendar rules, from research to NinjaTrader

The other notebooks in this repository trade at daily closes with a one-day lag. Retail platforms such as NinjaTrader work on
**bars** with open, high, low and close, and their strategies place **stop and limit orders that fill inside the bar**: a breakout
buys when the high crosses a level, a protective stop sells when the low reaches it, and the platform decides in which order the
two happened. This notebook uses the bar engine of `backtest_engine.bars` (C++, `cpp/bars.hpp`), which reproduces NinjaTrader 8's
historical fill model (`Calculate.OnBarClose`, Standard order fill resolution), to study four rules that retail traders deploy on
such platforms, and then exports them as NinjaScript strategies whose platform trade list can be reconciled with the backtest.

| Rule | Source | Bars | Orders |
| --- | --- | --- | --- |
| Donchian channel breakout with an ATR stop and a channel trailing exit ("Turtle system 1") | Faith (2007); Clenow (2013) | daily | stop entries, stop exits, reversals |
| RSI(2) mean reversion above the 200-day average, exit above the 5-day average | Connors and Alvarez (2009) | daily | market orders |
| Turn of the month: long from the open of the last weekday to the open after the third trading day | Lakonishok and Smidt (1988); McConnell and Xu (2008) | daily | market orders |
| Opening range breakout: first-bar direction, stop at the first bar's extreme, target in R multiples, flat at the close | Zarattini and Aziz (2023) | 5-minute | market entry, stop, target, session close |

The questions, in the order of the sections: how the execution model works and why it matters (section 2); whether each rule earns
anything after costs, how much depends on its parameters, and what survives once every configuration tried is accounted for
(sections 3-6, with grids, PBO, deflated Sharpe ratios and the SPA and Romano-Wolf tests of `backtest_engine.validation`); how much
of a bar-level backtest is an artefact of filling stop orders at the stop price (section 6); and how to carry a rule to the platform
and check that it trades the same way (section 7).

**Data.** Daily and 5-minute bars with open, high, low and close are not available from the official sources used elsewhere in this
repository, so this notebook works on two kinds of input: the user's own platform exports, when the files
`data/ninjatrader/daily.Last.txt` and `data/ninjatrader/intraday.Last.txt` exist (NinjaTrader *Tools > Historical Data > Export*;
never committed), and otherwise **synthetic bars** (`backtest_engine.synthetic`, seeded, clearly labelled, with a weak trend component
for the daily bars and none for the intraday bars). The calendar and RSI rules need closes only, so section 5 tests them on
**official data**: the Nasdaq Composite (FRED `NASDAQCOM`, daily closes since 1971) and the S&P 500 (FRED `SP500`, last ten years),
with the open of a day approximated by the previous close.

**Costs and sizing.** Micro E-mini S&P 500 contract specifications (point value 5, tick 0.25), 0.62 per contract and side in
commissions and one tick of slippage on market and stop fills, as a NinjaTrader commission template and the strategy's `Slippage`
property would charge; the breakout and the opening range breakout size each trade so that the loss at the stop is 1,000 (1% of a
100,000 account), the other rules trade one contract. Index results in section 5 are in returns with 5 basis points per unit of turnover.

**Caveats.** Synthetic bars show what the pipeline does, not what markets do; the numbers of sections 3, 4 and 6 are only evidence
about the data when the platform files are present. The fill model is a *specification* of NinjaTrader's documented behaviour that
still has to be confirmed on the reader's installation (section 7 explains how). Every result is hypothetical and not investment advice.

In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "backtest_engine").is_dir())
try:
    import backtest_engine
except ImportError:  # not installed: use the source folder (the extension must be built in place)
    sys.path.insert(0, str(ROOT / "scripts" / "backtest_engine"))
    import backtest_engine
from backtest_engine import bars as bl, data, metrics as mt, ninjatrader as nt, synthetic, validation as vl

core = backtest_engine.require_cpp()

DATA_MODE = os.environ.get("BACKTEST_ENGINE_DATA_MODE", "official")   # "official" (FRED closes, platform files) or "synthetic"
NT_DAILY = ROOT / "data" / "ninjatrader" / "daily.Last.txt"           # optional NinjaTrader exports (never committed)
NT_INTRADAY = ROOT / "data" / "ninjatrader" / "intraday.Last.txt"
SEED = 20260929
ES = bl.INSTRUMENTS["MES"]                                            # point value 5, tick 0.25, 0.62 per side, 1 tick slippage
RISK_PER_TRADE = 1000.0                                               # loss at the ATR stop, in currency (about 1% of 100,000)
IN_SAMPLE_FRACTION = 0.5                                              # first half of the bars in sample, second half out of sample
COST_BP = 5.0                                                         # index studies: basis points per unit of turnover
OUTPUT_DIR = ROOT / "outputs" / "bar_strategies"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.width", 160)
from matplotlib.colors import LinearSegmentedColormap
from backtest_engine.figstyle import (MUTED, NEGATIVE, NOTEBOOK, ORDINAL, SERIES, figure_headline, headline,
                                      label_ends, notebook_style)
DIVERGING = LinearSegmentedColormap.from_list("diverging", [SERIES[1], NOTEBOOK["neutral"], SERIES[0]])  # orange < 0 < blue
SEQUENTIAL = LinearSegmentedColormap.from_list("sequential", ORDINAL)
notebook_style()


def report(result, capital=None):
    s = bl.trade_statistics(result, capital)
    keys = ["net profit", "profit factor", "trades", "win rate", "average trade", "average win", "average loss",
            "largest loss", "max drawdown", "commissions", "annual Sharpe (daily P&L)"]
    return pd.Series({k: s[k] for k in keys}, name="value")


def split(index):
    cut = index[int(len(index) * IN_SAMPLE_FRACTION)]
    return index < cut, cut


print(f"backtest_engine {backtest_engine.__version__} | data mode: {DATA_MODE} | threads: {os.cpu_count()}")

backtest_engine 0.1.0 | data mode: official | threads: 4


## 1. Data

Three inputs: daily bars for the breakout, RSI and calendar rules; 5-minute bars of a regular session (09:30-16:00, stamped at the
bar's close as the platform does) for the opening range breakout; and, in official mode, index closes from FRED for section 5.

In [2]:
sources = []
if DATA_MODE == "official" and NT_DAILY.exists():
    daily = nt.read_export(NT_DAILY)
    sources.append(f"daily bars: NinjaTrader export {NT_DAILY.name} ({len(daily):,} bars)")
else:
    daily = synthetic.daily_bars(n=6500, start="2000-01-03", seed=SEED, level=4000.0, tick_size=0.25, annual_vol=0.18,
                                 trend_sharpe=0.6)
    sources.append(daily.attrs["source"] + ": driftless random walk with regime-switching trends of Sharpe ratio 0.6")
if DATA_MODE == "official" and NT_INTRADAY.exists():
    intraday = bl.restrict_to_session(nt.read_export(NT_INTRADAY), "09:30", "16:00")
    sources.append(f"intraday bars: NinjaTrader export {NT_INTRADAY.name} ({len(intraday):,} bars)")
else:
    intraday = synthetic.intraday_bars(n_days=750, start="2018-01-02", seed=SEED + 1, level=4000.0, tick_size=0.25,
                                       annual_vol=0.18, steps_per_bar=10)
    sources.append(intraday.attrs["source"] + ": martingale with overnight gaps (no intraday momentum)")
session = bl.session_ids(intraday.index)

closes = {}
if DATA_MODE == "official":
    try:
        fred = data.load_fred(["NASDAQCOM", "SP500"], start="1971-02-05", end="2025-12-31")
        closes = {"Nasdaq Composite": fred["NASDAQCOM"].dropna(), "S&P 500": fred["SP500"].dropna()}
        sources.append("index closes: Nasdaq Composite (FRED NASDAQCOM) and S&P 500 (FRED SP500)")
    except OSError as err:
        print(f"WARNING: FRED not reachable ({err}); section 5 runs on the synthetic daily closes.")
if not closes:
    closes = {"synthetic index": daily["close"].rename("synthetic")}
    sources.append("index closes: the synthetic daily bars' closes")
print("Sources:", *sources, sep="\n  ")
print(f"daily bars: {len(daily):,} from {daily.index[0]:%Y-%m-%d} to {daily.index[-1]:%Y-%m-%d}; "
      f"intraday bars: {len(intraday):,} in {session.max() + 1:,} sessions from {intraday.index[0]:%Y-%m-%d} to {intraday.index[-1]:%Y-%m-%d}")
for name, c in closes.items():
    print(f"{name}: {len(c):,} closes from {c.index[0]:%Y-%m-%d} to {c.index[-1]:%Y-%m-%d}")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(daily.index, daily["close"], color=SERIES[0], lw=1.0)
axes[0].set_yscale("log")
headline(axes[0], "Daily bars used by the breakout, RSI and calendar rules", "close, log scale")
day = intraday[session == session[len(intraday) // 2]]
up = day["close"] >= day["open"]
axes[1].vlines(day.index, day["low"], day["high"], color=MUTED, lw=0.8)
axes[1].vlines(day.index[up], day.loc[up, "open"], day.loc[up, "close"], color=SERIES[0], lw=3.0)
axes[1].vlines(day.index[~up], day.loc[~up, "open"], day.loc[~up, "close"], color=SERIES[1], lw=3.0)
axes[1].axvspan(day.index[0] - pd.Timedelta(minutes=5), day.index[0], color=NOTEBOOK["neutral"], zorder=0)
headline(axes[1], f"One session of 5-minute bars ({day.index[0]:%Y-%m-%d})", "open-close bodies (blue up, orange down), high-low wicks; first bar shaded")
import matplotlib.dates as mdates
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
fig.savefig(OUTPUT_DIR / "data.png", dpi=150)

Sources:
  synthetic daily bars (seed 20260929), not market data: driftless random walk with regime-switching trends of Sharpe ratio 0.6
  synthetic 5-minute bars (seed 20260930), not market data: martingale with overnight gaps (no intraday momentum)
  index closes: Nasdaq Composite (FRED NASDAQCOM) and S&P 500 (FRED SP500)
daily bars: 6,500 from 2000-01-03 to 2024-11-29; intraday bars: 58,500 in 750 sessions from 2018-01-02 to 2020-11-16
Nasdaq Composite: 13,844 closes from 1971-02-05 to 2025-12-31
S&P 500: 2,327 closes from 2016-09-29 to 2025-12-31


## 2. The execution model

`cpp/bars.hpp` simulates an *order plan*, one row per bar, the way NinjaTrader 8 fills orders on historical bars with
`Calculate.OnBarClose` and the Standard fill resolution:

| Order decided at the close of bar $t$ | Fill during bar $t+1$ |
| --- | --- |
| market | at the open (plus slippage) |
| stop | at the stop price when the range reaches it; at the open when the bar opens beyond it (plus slippage) |
| limit, profit target | at the limit price once the bar trades *through* it; at the open when it opens beyond it (`IsFillLimitOnTouch`: a touch is enough) |
| protective stop and target | attached when the entry fills, so they can fill on the entry bar; a stop in ticks is measured from the fill; a level already beyond the market when attached fills at once |
| exit on session close | at the close of the session's last bar |

When a stop and a target are both within the bar's range, the platform assumes the price moved **open, high, low, close** if the
open is closer to the high than to the low, and **open, low, high, close** otherwise, and fills the first order on that path (at the
same price a stop, which is reached, fills before a limit, which must be traded through). The example below shows the rule: the same
bar (high 4006, low 3994, close 4000) with a long entered at the open, a stop at 3998 and a target at 4002, both inside the range;
only the open differs, 4001 (nearer the high) or 3999 (nearer the low).

In [3]:
STOP, TARGET = 3998.0, 4002.0


def example(open_price):
    bars = pd.DataFrame([(4000.0, 4001.0, 3999.0, 4000.0), (open_price, 4006.0, 3994.0, 4000.0), (4000.0, 4001.0, 3999.0, 4000.0)],
                        columns=["open", "high", "low", "close"], index=pd.bdate_range("2024-01-01", periods=3))
    plan = bl.empty_plan(bars.index)
    plan.loc[bars.index[0], ["long_type", "long_stop", "long_target"]] = [1, STOP, TARGET]
    return bars, bl.run(bars, plan, ES.with_(commission=0.0, slippage_ticks=0.0)).trades.iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, open_price in zip(axes, (4001.0, 3999.0)):
    bars, trade = example(open_price)
    o, h, l, c = bars.iloc[1][["open", "high", "low", "close"]]
    path = [o, h, l, c] if (h - o) < (o - l) else [o, l, h, c]
    ax.plot(range(4), path, color=SERIES[0], marker="o", lw=1.5)
    for y, label, color in ((STOP, "stop", NEGATIVE), (TARGET, "target", SERIES[2])):
        ax.axhline(y, color=color, ls="--", lw=1.0)
        ax.text(3.05, y, label, color=color, va="center", fontsize=9)
    ax.set_xticks(range(4), ["open", "then", "then", "close"])
    ax.set_xlim(-0.3, 3.6)
    headline(ax, f"Open {o:g}: {trade['exit_reason']} fills first at {trade['exit_price']:g}",
             f"assumed path {' > '.join(f'{v:g}' for v in path)}; high {h:g}, low {l:g}")
fig.savefig(OUTPUT_DIR / "fill_path.png", dpi=150)
print("Both bars have the same high (4006), low (3994), close (4000), stop (3998) and target (4002): the open decides which order fills.")

Both bars have the same high (4006), low (3994), close (4000), stop (3998) and target (4002): the open decides which order fills.


## 3. Donchian channel breakout on daily bars

Buy stop one tick above the highest high of the last 20 bars (current bar included), sell stop one tick below the lowest low;
protective stop two ATR(20) from the fill, converted to an integer number of ticks; from the next bar the tighter of that stop and
the 10-bar channel; an opposite breakout reverses the position. Contracts are sized so that the loss at the ATR stop is at most 1,000
(at least one contract).

In [4]:
t0 = time.perf_counter()
plan_dc = bl.donchian_breakout_plan(daily, ES, entry_period=20, exit_period=10, atr_period=20, stop_atr=2.0, risk_per_trade=RISK_PER_TRADE)
dc = bl.run(daily, plan_dc, ES)
print(f"{len(daily):,} bars simulated in {1e3 * (time.perf_counter() - t0):.1f} ms (plan built in pandas, fills in C++)")
in_sample, cut = split(daily.index)
display(pd.DataFrame({"full sample": report(dc), "in sample": report(bl.run(daily[in_sample], plan_dc[in_sample], ES)),
                      "out of sample": report(bl.run(daily[~in_sample], plan_dc[~in_sample], ES))}))
print("Exit reasons:", dc.trades["exit_reason"].value_counts().to_dict())
display(dc.trades.head(6))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
equity = dc.equity
axes[0].plot(equity.index, equity, color=SERIES[0], lw=1.0)
axes[0].axvline(cut, color=NOTEBOOK["axis"], lw=1.0)
axes[0].text(cut, axes[0].get_ylim()[1], " out of sample", va="top", fontsize=8.5, color=NOTEBOOK["ink2"])
headline(axes[0], f"Net profit {equity.iloc[-1]:,.0f} after costs, {len(dc.trades)} trades", "cumulative P&L in currency, contracts sized to 1,000 at risk")
pnl = dc.trades["pnl"]
axes[1].hist(pnl[pnl < 0], bins=30, color=SERIES[1], label="losses")
axes[1].hist(pnl[pnl >= 0], bins=30, color=SERIES[0], label="gains")
axes[1].legend()
headline(axes[1], f"{(pnl > 0).mean():.0%} winners; average win {pnl[pnl > 0].mean():,.0f}, average loss {pnl[pnl < 0].mean():,.0f}",
         "distribution of trade P&L (currency)")
fig.savefig(OUTPUT_DIR / "donchian_baseline.png", dpi=150)

6,500 bars simulated in 24.1 ms (plan built in pandas, fills in C++)


,full sample,in sample,out of sample
net profit,"31,111.870","-27,138.330","57,177.710"
profit factor,1.131,0.723,1.406
trades,304.000,159.000,146.000
win rate,0.382,0.314,0.452
average trade,102.342,-170.681,391.628
average win,"2,318.728","1,418.310","3,000.862"
average loss,"-1,265.216",-899.577,"-1,760.990"
largest loss,"-4,192.490","-2,107.490","-4,192.490"
max drawdown,"46,465.440","46,465.440","30,584.720"
commissions,386.880,207.080,181.040


Exit reasons: {'stop': 303, 'end of data': 1}


,entry_time,exit_time,entry_bar,exit_bar,side,qty,entry_price,exit_price,pnl,commission,mae,mfe,exit_reason,bars
trade,,,,,,,,,,,,,,
1,2000-02-01,2000-02-02,21,22,-1,1,"3,536.000","3,647.000",-556.240,1.240,111.000,0.000,stop,1
2,2000-02-07,2000-02-23,25,37,-1,1,"3,535.750","3,546.000",-52.490,1.240,22.000,160.000,stop,12
3,2000-02-28,2000-03-31,40,64,1,1,"3,674.500","3,739.250",322.510,1.240,50.750,223.000,stop,24
4,2000-04-10,2000-04-24,70,80,1,1,"3,898.000","3,860.000",-191.240,1.240,38.000,101.000,stop,10
5,2000-05-11,2000-05-18,93,98,1,1,"4,001.250","3,885.750",-578.740,1.240,115.500,74.000,stop,5
6,2000-06-06,2000-07-28,111,149,1,1,"4,075.750","4,475.750","1,998.760",1.240,0.250,580.500,stop,38


### Robustness grid, selection and multiple testing

54 configurations (entry channel 10-80 bars, exit channel 5-20 bars, stop 1.5-3 ATR) run in parallel in C++. The in-sample winner is
selected on the first half of the sample and evaluated on the second half; the probability of backtest overfitting (CSCV) and the
deflated Sharpe ratio account for the search, and the SPA test and the Romano-Wolf stepdown ask whether *any* configuration has a
positive expected P&L after the search.

In [5]:
grid_dc = [(e, x, s) for e in (10, 20, 30, 40, 55, 80) for x in (5, 10, 20) for s in (1.5, 2.0, 3.0)]
names_dc = [f"entry={e} exit={x} stop={s:g}" for e, x, s in grid_dc]
t0 = time.perf_counter()
plans = [bl.donchian_breakout_plan(daily, ES, entry_period=e, exit_period=x, atr_period=20, stop_atr=s, risk_per_trade=RISK_PER_TRADE) for e, x, s in grid_dc]
pnl_dc, n_trades_dc = bl.run_many(daily, plans, ES, names=names_dc)
print(f"{len(plans)} configurations x {len(daily):,} bars in {time.perf_counter() - t0:.2f} s")
warm = max(80, 20)
pnl_dc = pnl_dc.iloc[warm:]
in_sample = pnl_dc.index < cut
configs_dc = pd.DataFrame(grid_dc, columns=["entry", "exit", "stop"], index=names_dc)
configs_dc["trades"] = n_trades_dc
configs_dc["Sharpe full"] = pnl_dc.apply(mt.sharpe_ratio)
configs_dc["Sharpe in sample"] = pnl_dc[in_sample].apply(mt.sharpe_ratio)
configs_dc["Sharpe out of sample"] = pnl_dc[~in_sample].apply(mt.sharpe_ratio)
best_dc = configs_dc["Sharpe in sample"].idxmax()
pbo_dc = core.cscv(pnl_dc.to_numpy(), 16, 0)
trial_sharpes = pnl_dc[in_sample].apply(lambda c: c.mean() / c.std(ddof=1))
dsr_dc = mt.deflated_sharpe_ratio(pnl_dc.loc[in_sample, best_dc], trial_sharpes)
snoop_dc = vl.SnoopingTest(pnl_dc, n_boot=5000, seed=SEED)
snoop_dc_oos = vl.SnoopingTest(pnl_dc[~in_sample], n_boot=5000, seed=SEED)
display(pd.Series({"in-sample winner": best_dc, "Sharpe in sample": configs_dc.loc[best_dc, "Sharpe in sample"],
                   "Sharpe out of sample": configs_dc.loc[best_dc, "Sharpe out of sample"],
                   "median out-of-sample Sharpe of the grid": configs_dc["Sharpe out of sample"].median(),
                   "share of configurations with positive full-sample Sharpe": (configs_dc["Sharpe full"] > 0).mean(),
                   "PBO (CSCV, 16 blocks)": pbo_dc["pbo"], "deflated Sharpe ratio of the winner": dsr_dc["dsr"],
                   "effective number of trials (bootstrap)": snoop_dc.effective_trials(),
                   "SPA p (consistent), full sample": snoop_dc.spa()["p_consistent"],
                   "SPA p (consistent), out of sample": snoop_dc_oos.spa()["p_consistent"],
                   "Romano-Wolf rejections at 5%, full sample": int(snoop_dc.romano_wolf()["reject"].sum())}, name="value").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
heat = configs_dc.pivot_table(index="stop", columns="entry", values="Sharpe full", aggfunc="mean")
im = axes[0].imshow(heat.to_numpy(), aspect="auto", cmap=DIVERGING, vmin=-1, vmax=1)
axes[0].set_xticks(range(len(heat.columns)), heat.columns)
axes[0].set_yticks(range(len(heat.index)), [f"{v:g}" for v in heat.index])
axes[0].set(xlabel="entry channel (bars)", ylabel="stop (ATR)")
axes[0].grid(False)
fig.colorbar(im, ax=axes[0], label="Sharpe ratio")
headline(axes[0], f"{(configs_dc['Sharpe full'] > 0).mean():.0%} of {len(configs_dc)} configurations have a positive Sharpe ratio", "full-sample Sharpe ratio averaged over the exit channel")
axes[1].scatter(configs_dc["Sharpe in sample"], configs_dc["Sharpe out of sample"], c=configs_dc["entry"], cmap=SEQUENTIAL, s=22, edgecolors=NOTEBOOK["surface"], linewidths=0.4)
axes[1].axhline(0, color=NOTEBOOK["axis"]); axes[1].axvline(0, color=NOTEBOOK["axis"])
axes[1].scatter(configs_dc.loc[best_dc, "Sharpe in sample"], configs_dc.loc[best_dc, "Sharpe out of sample"], s=90, facecolors="none", edgecolors=NEGATIVE, linewidths=1.5)
axes[1].set(xlabel="Sharpe in sample", ylabel="Sharpe out of sample")
headline(axes[1], f"In-sample winner: {configs_dc.loc[best_dc, 'Sharpe in sample']:.2f} in sample, {configs_dc.loc[best_dc, 'Sharpe out of sample']:.2f} out of sample (circled)",
         f"PBO {pbo_dc['pbo']:.2f}; colour: entry channel length")
fig.savefig(OUTPUT_DIR / "donchian_grid.png", dpi=150)

54 configurations x 6,500 bars in 1.06 s


,value
in-sample winner,entry=30 exit=20 stop=2
Sharpe in sample,0.038
Sharpe out of sample,0.157
median out-of-sample Sharpe of the grid,0.270
share of configurations with positive full-sample Sharpe,0.852
"PBO (CSCV, 16 blocks)",0.303
deflated Sharpe ratio of the winner,0.171
effective number of trials (bootstrap),3.555
"SPA p (consistent), full sample",0.212
"SPA p (consistent), out of sample",0.071


## 4. RSI(2) mean reversion on daily bars

Buy at the next open when the close is above its 200-bar average and NinjaTrader's RSI(2) is below 10; sell at the next open once the
close is above its 5-bar average. Market orders only, one contract. The grid varies the entry level (5-30) and the exit average
(3-10 bars); the trend filter stays at 200 bars.

In [6]:
plan_rsi = bl.rsi2_plan(daily, ES, rsi_period=2, entry_level=10.0, exit_ma=5, trend_ma=200)
rsi = bl.run(daily, plan_rsi, ES)
in_sample, cut = split(daily.index)
display(pd.DataFrame({"full sample": report(rsi), "in sample": report(bl.run(daily[in_sample], plan_rsi[in_sample], ES)),
                      "out of sample": report(bl.run(daily[~in_sample], plan_rsi[~in_sample], ES))}))
grid_rsi = [(lv, x) for lv in (5.0, 10.0, 15.0, 20.0, 25.0, 30.0) for x in (3, 5, 10)]
names_rsi = [f"level={lv:g} exit={x}" for lv, x in grid_rsi]
pnl_rsi, n_trades_rsi = bl.run_many(daily, [bl.rsi2_plan(daily, ES, entry_level=lv, exit_ma=x) for lv, x in grid_rsi], ES, names=names_rsi)
pnl_rsi = pnl_rsi.iloc[200:]
in_sample = pnl_rsi.index < cut
configs_rsi = pd.DataFrame(grid_rsi, columns=["level", "exit"], index=names_rsi)
configs_rsi["trades"] = n_trades_rsi
configs_rsi["Sharpe in sample"] = pnl_rsi[in_sample].apply(mt.sharpe_ratio)
configs_rsi["Sharpe out of sample"] = pnl_rsi[~in_sample].apply(mt.sharpe_ratio)
configs_rsi["Sharpe full"] = pnl_rsi.apply(mt.sharpe_ratio)
best_rsi = configs_rsi["Sharpe in sample"].idxmax()
snoop_rsi = vl.SnoopingTest(pnl_rsi, n_boot=5000, seed=SEED)
display(pd.Series({"in-sample winner": best_rsi, "Sharpe in sample": configs_rsi.loc[best_rsi, "Sharpe in sample"],
                   "Sharpe out of sample": configs_rsi.loc[best_rsi, "Sharpe out of sample"],
                   "PBO (CSCV, 16 blocks)": core.cscv(pnl_rsi.to_numpy(), 16, 0)["pbo"],
                   "SPA p (consistent), full sample": snoop_rsi.spa()["p_consistent"],
                   "Romano-Wolf rejections at 5%": int(snoop_rsi.romano_wolf()["reject"].sum())}, name="value").to_frame())
display(configs_rsi.pivot(index="level", columns="exit", values="Sharpe full").rename_axis(columns="exit average (bars)"))

,full sample,in sample,out of sample
net profit,-508.350,"-1,964.110","1,455.760"
profit factor,0.994,0.928,1.025
trades,165.000,89.000,76.000
win rate,0.624,0.618,0.632
average trade,-3.081,-22.069,19.155
average win,828.529,462.715,"1,247.692"
average loss,"-1,384.627",-806.277,"-2,086.910"
largest loss,"-11,764.990","-4,536.240","-11,764.990"
max drawdown,"22,046.700","11,181.730","22,046.700"
commissions,204.600,110.360,94.240


,value
in-sample winner,level=15 exit=3
Sharpe in sample,0.471
Sharpe out of sample,0.066
"PBO (CSCV, 16 blocks)",0.561
"SPA p (consistent), full sample",0.411
Romano-Wolf rejections at 5%,0


exit average (bars),3,5,10
level,,,
5.000,0.015,-0.088,-0.140
10.000,0.089,-0.005,-0.028
15.000,0.203,0.152,0.119
20.000,0.096,0.067,0.055
25.000,0.146,0.100,0.082
30.000,0.237,0.150,0.072


## 5. Calendar and RSI rules on official index closes

The turn-of-the-month effect (returns from the last trading day of a month to its third trading day) is one of the most persistent
calendar anomalies: Lakonishok and Smidt (1988) found it in ninety years of Dow Jones data, McConnell and Xu (2008) in US and
international indices through 2005. It needs closes only, so it can be tested here on official data: the Nasdaq Composite since 1971
(the bars' open is the previous close, so "buy at the open of the last weekday" is buying at the second-last close) with 5 basis points
per unit of turnover. The RSI(2) rule is tested on the same closes: its fill at the next open is then the signal day's own close,
which is the rule's original form (Connors and Alvarez enter at the close of the signal day). In-sample period 1971-2000, out of sample 2001-2025; the rule (1 day before, 3 after) was fixed before
looking at the data, and a 20-configuration grid (0-3 days before, 1-5 after) is tested with SPA and Romano-Wolf.

In [7]:
INDEX = bl.INSTRUMENTS["index"]
IS_END = pd.Timestamp("2000-12-31")
tom_grid = [(b, a) for b in (0, 1, 2, 3) for a in (1, 2, 3, 4, 5)]
tom_names = [f"before={b} after={a}" for b, a in tom_grid]
rows, curves, tests = {}, {}, {}
for name, close in closes.items():
    ibars = bl.bars_from_closes(close)
    prev_close = ibars["close"].shift(1)

    def returns_of(plan):
        r = bl.run(ibars, plan, INDEX)
        turnover = r.position.diff().abs().fillna(0.0)
        return (r.pnl / prev_close - COST_BP * 1e-4 * turnover).fillna(0.0), r.position.shift(1).fillna(0.0) != 0

    base, held = returns_of(bl.turn_of_month_plan(ibars, INDEX, days_before=1, days_after=3))
    buy_hold = close.pct_change().fillna(0.0)
    pre, post = base.index <= IS_END, base.index > IS_END
    for label, mask in (("full sample", np.ones(len(base), bool)), ("to 2000", pre), ("2001-2025", post)):
        if mask.sum() > 300:
            s = mt.performance_summary(base[mask])
            bh = mt.performance_summary(buy_hold[mask])
            rows[(name, label)] = {"days in market": float(held[mask].mean()), "annual mean": s["annual mean"],
                                   "annual volatility": s["annual volatility"], "Sharpe": s["Sharpe"], "max drawdown": s["max drawdown"],
                                   "buy-and-hold Sharpe": bh["Sharpe"], "buy-and-hold annual mean": bh["annual mean"]}
    curves[name] = (base, held, buy_hold)
    if len(base) > 2000:
        grid = pd.DataFrame({n: returns_of(bl.turn_of_month_plan(ibars, INDEX, days_before=b, days_after=a))[0] for n, (b, a) in zip(tom_names, tom_grid)})
        tests[name] = (grid, vl.SnoopingTest(grid, n_boot=5000, seed=SEED), vl.SnoopingTest(grid[post], n_boot=5000, seed=SEED))
table = pd.DataFrame(rows).T
table.index.names = ["index", "period"]
display(table)

main = next(iter(closes))
base, held, buy_hold = curves[main]
boot = mt.bootstrap_sharpe(base[base.index > IS_END].to_numpy(), mean_block=20, n_boot=10_000, seed=SEED)
print(f"{main}, 2001-2025: turn-of-month Sharpe {mt.sharpe_ratio(base[base.index > IS_END]):.2f}, "
      f"bootstrap 95% interval {np.percentile(boot, 2.5):.2f} to {np.percentile(boot, 97.5):.2f}")
if main in tests:
    grid, full_test, oos_test = tests[main]
    rw = full_test.romano_wolf()
    display(pd.Series({"configurations": len(tom_grid), "best (full sample)": full_test.naive()["best"],
                       "naive p of the best": full_test.naive()["p"], "SPA p (consistent), full sample": full_test.spa()["p_consistent"],
                       "SPA p (consistent), 2001-2025": oos_test.spa()["p_consistent"],
                       "Romano-Wolf rejections at 5%, full sample": int(rw["reject"].sum()),
                       "Romano-Wolf rejections at 5%, 2001-2025": int(oos_test.romano_wolf()["reject"].sum()),
                       "adjusted p of the a-priori rule (before=1 after=3)": rw.loc["before=1 after=3", "p_adjusted"]}, name="value").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
wealth_tom = (1 + base).cumprod()
wealth_rest = (1 + buy_hold.where(~held, 0.0)).cumprod()
wealth_bh = (1 + buy_hold).cumprod()
for series, label, color in ((wealth_tom, "turn-of-month days (after costs)", SERIES[0]), (wealth_rest, "all other days", SERIES[1]), (wealth_bh, "buy and hold", MUTED)):
    axes[0].plot(series.index, series, color=color, lw=1.0, label=label)
axes[0].set_yscale("log")
axes[0].legend(loc="upper left")
share_days = held.mean()
share_growth = np.log(wealth_tom.iloc[-1]) / (np.log(wealth_tom.iloc[-1]) + np.log(wealth_rest.iloc[-1]))
headline(axes[0], f"{main}: {share_days:.0%} of the days produced {share_growth:.0%} of the log growth", "growth of 1, log scale; the rule pays 5 bp per unit of turnover, the other days nothing")
yearly = base.groupby(base.index.year).apply(lambda r: (1 + r).prod() - 1)
axes[1].bar(yearly.index, 100 * yearly, color=np.where(yearly >= 0, SERIES[0], SERIES[1]))
headline(axes[1], f"Positive in {(yearly > 0).mean():.0%} of the years", "yearly compounded return of the turn-of-month rule (%)")
fig.savefig(OUTPUT_DIR / "turn_of_month.png", dpi=150)

days in market  annual mean  annual volatility  Sharpe  max drawdown  buy-and-hold Sharpe  buy-and-hold annual mean
index            period                                                                                                                          
Nasdaq Composite full sample           0.185        0.046              0.089   0.517         0.300                0.592                     0.120
                 to 2000               0.184        0.067              0.075   0.896         0.235                0.719                     0.121
                 2001-2025             0.186        0.021              0.103   0.200         0.300                0.498                     0.118
S&P 500          full sample           0.185        0.012              0.076   0.162         0.113                0.774                     0.142
                 2001-2025             0.185        0.012              0.076   0.162         0.113                0.774                     0.142

Nasdaq Composite, 2001-2025: turn-of-month Sharpe 0.20, bootstrap 95% interval -0.19 to 0.57


,value
configurations,20
best (full sample),before=3 after=4
naive p of the best,0.000
"SPA p (consistent), full sample",0.001
"SPA p (consistent), 2001-2025",0.202
"Romano-Wolf rejections at 5%, full sample",19
"Romano-Wolf rejections at 5%, 2001-2025",0
adjusted p of the a-priori rule (before=1 after=3),0.001


In [8]:
rows = {}
rsi_grid = [(lv, x) for lv in (5.0, 10.0, 15.0, 20.0, 25.0, 30.0) for x in (3, 5, 10)]
rsi_names = [f"level={lv:g} exit={x}" for lv, x in rsi_grid]
for name, close in closes.items():
    ibars = bl.bars_from_closes(close)
    prev_close = ibars["close"].shift(1)

    def returns_of(plan):
        r = bl.run(ibars, plan, INDEX)
        return (r.pnl / prev_close - COST_BP * 1e-4 * r.position.diff().abs().fillna(0.0)).fillna(0.0), r.position.shift(1).fillna(0.0) != 0

    base, held = returns_of(bl.rsi2_plan(ibars, INDEX, entry_level=10.0, exit_ma=5, trend_ma=200))
    base, held = base.iloc[200:], held.iloc[200:]
    pre, post = base.index <= IS_END, base.index > IS_END
    for label, mask in (("full sample", np.ones(len(base), bool)), ("to 2000", pre), ("2001-2025", post)):
        if mask.sum() > 300:
            s = mt.performance_summary(base[mask])
            rows[(name, label)] = {"days in market": float(held[mask].mean()), "annual mean": s["annual mean"],
                                   "annual volatility": s["annual volatility"], "Sharpe": s["Sharpe"], "max drawdown": s["max drawdown"]}
    if name == main and len(base) > 2000:
        grid = pd.DataFrame({n: returns_of(bl.rsi2_plan(ibars, INDEX, entry_level=lv, exit_ma=x, trend_ma=200))[0] for n, (lv, x) in zip(rsi_names, rsi_grid)}).iloc[200:]
        rsi_test, rsi_test_oos = vl.SnoopingTest(grid, n_boot=5000, seed=SEED), vl.SnoopingTest(grid[post], n_boot=5000, seed=SEED)
table = pd.DataFrame(rows).T
table.index.names = ["index", "period"]
print("RSI(2) rule on index closes (entry and exit at the signal day's close)")
display(table)
display(pd.Series({"configurations": len(rsi_grid), "best (full sample)": rsi_test.naive()["best"], "naive p of the best": rsi_test.naive()["p"],
                   "SPA p (consistent), full sample": rsi_test.spa()["p_consistent"], "SPA p (consistent), 2001-2025": rsi_test_oos.spa()["p_consistent"],
                   "Romano-Wolf rejections at 5%, full sample": int(rsi_test.romano_wolf()["reject"].sum()),
                   "adjusted p of the a-priori rule (level=10 exit=5)": rsi_test.romano_wolf().loc["level=10 exit=5", "p_adjusted"]}, name="value").to_frame())

RSI(2) rule on index closes (entry and exit at the signal day's close)


days in market  annual mean  annual volatility  Sharpe  max drawdown
index            period                                                                           
Nasdaq Composite full sample           0.124        0.000              0.076   0.000         0.769
                 to 2000               0.143       -0.034              0.080  -0.432         0.769
                 2001-2025             0.102        0.040              0.070   0.574         0.156
S&P 500          full sample           0.114        0.028              0.065   0.433         0.157
                 2001-2025             0.114        0.028              0.065   0.433         0.157

,value
configurations,18
best (full sample),level=10 exit=10
naive p of the best,0.448
"SPA p (consistent), full sample",0.796
"SPA p (consistent), 2001-2025",0.000
"Romano-Wolf rejections at 5%, full sample",0
adjusted p of the a-priori rule (level=10 exit=5),0.835


## 6. Opening range breakout on 5-minute bars, and the cost of bar-level fills

At the close of the first 5-minute bar of the session: if it closed above its open, buy at the next open with a stop at the bar's low and
a target ten times the distance from the close to the stop; the mirror image if it closed below; flat at the session's last close. This is the
rule of Zarattini and Aziz (2023), who report a 33% annual alpha on QQQ in 2016-2023. Most trades end at the stop, so the result depends
on **how the stop is filled**: a bar-level backtest (this engine, and NinjaTrader's Standard resolution) fills a stop order at the stop
price, whereas the price actually jumps through it. The second part of the section measures that optimism on a martingale, where the true
expected profit is zero: the same rule is run on synthetic 5-minute bars whose intrabar path has 10 or 50 steps, with 0, 1 and 2 ticks of
slippage, over several seeds.

In [9]:
ORB = ES.with_(exit_on_session_close=True)
plan_orb = bl.opening_range_breakout_plan(intraday, ORB, session, r_multiple=10.0, risk_per_trade=RISK_PER_TRADE)
orb = bl.run(intraday, plan_orb, ORB, session)
display(report(orb).to_frame("baseline (1 tick slippage, 0.62 commission)"))
print("Exit reasons:", orb.trades["exit_reason"].value_counts().to_dict())

t0 = time.perf_counter()
rows = []
null_seeds = range(6)
for steps in (10, 50):
    for seed in null_seeds:
        b = synthetic.intraday_bars(n_days=300, start="2018-01-02", seed=1000 + seed, level=4000.0, tick_size=0.25, steps_per_bar=steps)
        s_ = bl.session_ids(b.index)
        p_ = bl.opening_range_breakout_plan(b, ORB, s_, r_multiple=10.0, risk_per_trade=RISK_PER_TRADE)
        for slip in (0.0, 1.0, 2.0):
            r_ = bl.run(b, p_, ORB.with_(slippage_ticks=slip, commission=0.0), s_)
            st = bl.trade_statistics(r_)
            rows.append({"steps per bar": steps, "slippage (ticks)": slip, "seed": seed, "Sharpe": st["annual Sharpe (daily P&L)"],
                         "net profit": st["net profit"], "stopped": (r_.trades["exit_reason"] == "stop").mean()})
null = pd.DataFrame(rows)
print(f"null study: {len(rows)} runs in {time.perf_counter() - t0:.0f} s")
summary = null.groupby(["steps per bar", "slippage (ticks)"]).agg(**{"mean Sharpe": ("Sharpe", "mean"), "s.e.": ("Sharpe", lambda x: x.std() / np.sqrt(len(x))),
                                                                     "mean net profit": ("net profit", "mean"), "share stopped": ("stopped", "mean")})
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
eq = orb.daily_pnl().cumsum()
axes[0].plot(eq.index, eq, color=SERIES[0], lw=1.0)
headline(axes[0], f"Net profit {eq.iloc[-1]:,.0f} on {len(orb.trades)} trades, {(orb.trades['exit_reason'] == 'stop').mean():.0%} stopped",
         "cumulative daily P&L (currency) of the opening range breakout on the intraday bars")
for steps, color in ((10, SERIES[1]), (50, SERIES[0])):
    sub = summary.loc[steps]
    axes[1].errorbar(sub.index, sub["mean Sharpe"], yerr=2 * sub["s.e."], color=color, marker="o", capsize=3, label=f"{steps} steps per bar")
axes[1].axhline(0, color=NOTEBOOK["axis"])
axes[1].legend()
axes[1].set(xlabel="slippage (ticks) on market and stop fills", ylabel="annual Sharpe ratio")
headline(axes[1], f"On a martingale the bar-level backtest shows a Sharpe ratio of {summary.loc[(10, 0.0), 'mean Sharpe']:.1f} without slippage",
         "mean over 6 seeds of 300 sessions, 2 standard errors; the coarser the true path, the larger the stop-fill optimism")
fig.savefig(OUTPUT_DIR / "orb_fill_optimism.png", dpi=150)

,"baseline (1 tick slippage, 0.62 commission)"
net profit,"-85,154.650"
profit factor,0.895
trades,735.000
win rate,0.133
average trade,-115.857
average win,"7,387.579"
average loss,"-1,270.231"
largest loss,"-3,992.000"
max drawdown,"154,999.570"
commissions,"67,623.400"


Exit reasons: {'stop': 637, 'target': 57, 'session close': 41}


null study: 36 runs in 10 s


mean Sharpe  s.e.  mean net profit  share stopped
steps per bar slippage (ticks)                                                   
10            0.000                   1.117 0.301       70,051.667          0.857
              1.000                   0.381 0.326       25,671.250          0.857
              2.000                  -0.341 0.352      -18,709.167          0.857
50            0.000                  -0.475 0.523      -18,630.000          0.894
              1.000                  -1.196 0.532      -57,519.375          0.894
              2.000                  -1.904 0.540      -96,408.750          0.894

## 7. Export to NinjaTrader and reconciliation

`backtest_engine.ninjatrader` renders each rule as a NinjaScript strategy with the researched parameters as defaults; the files with the
library defaults are committed in [`scripts/ninjatrader_strategies/`](../../scripts/ninjatrader_strategies/README.md), those with this
notebook's configurations are written to `outputs/ninjatrader/`, next to the bars used here in the platform's import format. The parity
check on the reader's installation is:

1. import the bars (or export the platform's own bars and rerun this notebook on them, see section 1);
2. compile the strategy, apply a commission template with 0.62 per contract and side (MES), run the Strategy Analyzer with the same slippage;
3. export the *Trades* tab and call `ninjatrader.reconcile_trades` on it and on `result.trades`.

The last cell runs the reconciliation on a copy of our own trade list in the platform's export format, which shows the report; a real
platform export will differ where the platform's fill logic departs from the documented model, and those differences are what the check
is for.

In [10]:
NT_DIR = ROOT / "outputs" / "ninjatrader"
NT_DIR.mkdir(parents=True, exist_ok=True)
params = {"donchian_breakout": {"entry_period": 20, "exit_period": 10, "atr_period": 20, "stop_atr": 2.0, "risk_per_trade": RISK_PER_TRADE},
          "rsi2": {"entry_level": 10.0, "exit_ma": 5, "trend_ma": 200}, "turn_of_month": {"days_before": 1, "days_after": 3},
          "opening_range_breakout": {"r_multiple": 10.0, "risk_per_trade": RISK_PER_TRADE}}
written = nt.write_strategies(NT_DIR, params, {name: ES for name in params})
nt.write_export(daily, NT_DIR / "daily.Last.txt")
nt.write_export(intraday, NT_DIR / "intraday.Last.txt")
dc.trades.to_csv(NT_DIR / "donchian_trades_python.csv")
print("Written:", *[p.name for p in written], "daily.Last.txt", "intraday.Last.txt", "donchian_trades_python.csv", sep="\n  ")

# The trade list as the platform exports it, built from our own trades with one exit price moved by a tick.
closed = dc.trades[dc.trades["exit_reason"] != "end of data"]
rows = []
for k, tr in closed.iterrows():
    exit_price = tr["exit_price"] + (ES.tick_size if k == closed.index[3] else 0.0)
    pnl = tr["side"] * tr["qty"] * (exit_price - tr["entry_price"]) * ES.point_value - tr["commission"]
    rows.append({"Trade-#": k, "Instrument": "MES", "Account": "Backtest", "Strategy": "BtDonchianBreakout",
                 "Market pos.": "Long" if tr["side"] > 0 else "Short", "Qty": tr["qty"], "Entry price": f"{tr['entry_price']:.2f}",
                 "Exit price": f"{exit_price:.2f}", "Entry time": tr["entry_time"].strftime("%m/%d/%Y %I:%M:%S %p"),
                 "Exit time": tr["exit_time"].strftime("%m/%d/%Y %I:%M:%S %p"), "Entry name": "Long" if tr["side"] > 0 else "Short",
                 "Exit name": "Stop loss", "Profit": f"(${-pnl:,.2f})" if pnl < 0 else f"${pnl:,.2f}", "Cum. net profit": "$0.00",
                 "Commission": f"${tr['commission']:.2f}", "MAE": f"${tr['mae'] * ES.point_value:,.2f}", "MFE": f"${tr['mfe'] * ES.point_value:,.2f}",
                 "ETD": "$0.00", "Bars": tr["bars"]})
pd.DataFrame(rows).to_csv(NT_DIR / "donchian_trades_platform_format_example.csv", index=False)
theirs = nt.read_strategy_analyzer_trades(NT_DIR / "donchian_trades_platform_format_example.csv")
check = nt.reconcile_trades(dc.trades, theirs, tolerance="1D")
display(pd.Series(check["summary"], name="value").to_frame())
display(check["matched"].query("not prices_agree")[["trade_ours", "trade_theirs", "side", "entry_time_ours", "entry_price_ours", "entry_price_theirs",
                                                   "exit_price_ours", "exit_price_theirs", "pnl_diff"]])

Written:
  BtDonchianBreakout.cs
  BtRsi2MeanReversion.cs
  BtOpeningRangeBreakout.cs
  BtTurnOfMonth.cs
  daily.Last.txt
  intraday.Last.txt
  donchian_trades_python.csv


,value
ours,303.000
theirs,303.000
matched,303.000
matched with equal prices and quantities,302.000
only ours,0.000
only theirs,0.000
max |entry price difference|,0.000
max |exit price difference|,0.250
net profit ours,"22,593.110"
net profit theirs,"22,594.360"


,trade_ours,trade_theirs,side,entry_time_ours,entry_price_ours,entry_price_theirs,exit_price_ours,exit_price_theirs,pnl_diff
3,4,4,1,2000-04-10,"3,898.000","3,898.000","3,860.000","3,860.250",1.250


## 8. Interpretation guide

- **The execution model is part of the strategy.** A breakout rule is defined by *where* it buys, and a bar-level backtest defines it by
  assumption: fills at the stop price, a fixed intrabar path, no queue at the limit. Section 6 shows that these assumptions alone give a
  martingale a Sharpe ratio around one for a rule that is stopped out on most days. NinjaTrader's Standard fill resolution shares them;
  its *High* resolution (fills on a finer series) and a slippage setting of one or two ticks are the platform's remedies, and a strategy
  whose edge disappears with two ticks of slippage has no edge. The same optimism is smaller for rules that trade with market orders at
  the open (RSI, calendar) and for trend rules whose gains come from long holding periods.
- **Grids are read as a whole.** A broad region of positive Sharpe ratios in the breakout heat map is evidence about the data; an isolated
  hot spot is evidence about the search. The in-sample winner's fall out of sample, the PBO, the deflated Sharpe ratio and the SPA
  p-value quantify how much of the best number is selection.
- **Calendar effects are the easiest to trade and the hardest to explain.** The turn-of-the-month rule is in the market a fifth of the
  time with market orders once a month, so costs are negligible; the question is whether the effect persists after its publication
  (compare the periods in the table) and whether the a-priori window survives the family of neighbouring windows (Romano-Wolf adjusted
  p-value of the rule fixed in advance).
- **On synthetic data the numbers are a demonstration, not a result**: the daily bars have weak trends by construction, so the breakout
  finds something; the intraday bars are a martingale, so the opening range breakout should not. The pipeline is the same when the
  platform files are present, which is the intended use.
- **Parity is checked, not assumed.** The NinjaScript strategies re-submit their orders every bar, size and round prices exactly as the
  Python plans do, and the reconciliation reports every trade whose time, side, quantity or price differs. A difference points to a
  platform rule that the engine models differently (the README of `scripts/ninjatrader_strategies/` lists the documented ambiguities), and
  the engine, not the NinjaScript, is the place to fix it.

Figures are saved in `outputs/bar_strategies/`, the NinjaScript strategies and the platform-format bars in `outputs/ninjatrader/`.

### References

- Bailey, D. H., Borwein, J., Lopez de Prado, M. and Zhu, Q. J. (2017). The probability of backtest overfitting. *Journal of Computational Finance*, 20(4), 39-69.
- Bailey, D. H. and Lopez de Prado, M. (2014). The deflated Sharpe ratio. *Journal of Portfolio Management*, 40(5), 94-107.
- Clenow, A. (2013). *Following the Trend: Diversified Managed Futures Trading*. Wiley.
- Connors, L. and Alvarez, C. (2009). *Short Term Trading Strategies That Work*. TradingMarkets Publishing.
- Faith, C. (2007). *Way of the Turtle*. McGraw-Hill.
- Hansen, P. R. (2005). A test for superior predictive ability. *Journal of Business and Economic Statistics*, 23(4), 365-380.
- Lakonishok, J. and Smidt, S. (1988). Are seasonal anomalies real? A ninety-year perspective. *Review of Financial Studies*, 1(4), 403-425.
- McConnell, J. J. and Xu, W. (2008). Equity returns at the turn of the month. *Financial Analysts Journal*, 64(2), 49-64.
- NinjaTrader (2024). NinjaTrader 8 help guide: Historical order backfill logic; Order fill resolution; IsFillLimitOnTouch. https://ninjatrader.com/support/helpGuides/nt8/
- Romano, J. P. and Wolf, M. (2005). Stepwise multiple testing as formalized data snooping. *Econometrica*, 73(4), 1237-1282.
- Zarattini, C. and Aziz, A. (2023). Can day trading really be profitable? Evidence of sustainable long-term profits from opening range breakout day trading strategy vs. benchmark in the US stock market. SSRN working paper 4416622.
- Data: Nasdaq Composite Index (NASDAQCOM) and S&P 500 (SP500) via FRED, Federal Reserve Bank of St. Louis. Contract specifications: CME Group.